# AudioGen Interactive GPU Server Notebook
Runs the FastAPI speech synthesis server, Cloudflare quick tunnel, and idle watchdog on Kaggle.

In [ ]:
# Cell 1: Dependencies & Environment Setup
import os
import sys
from pathlib import Path
import shutil

# Resolve repository root across various execution environments:
# 1. Local execution: notebook located in server/ or repository root
# 2. Remote Kaggle execution: notebook executed in /kaggle/working
repo_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/kaggle/working/audiogen"),
    Path("/kaggle/working"),
]

for candidate in repo_candidates:
    if (candidate / "server").is_dir() and (candidate / "voices").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if (candidate / "src").is_dir() and str(candidate / "src") not in sys.path:
            sys.path.insert(0, str(candidate / "src"))
        req_file = candidate / "requirements.txt"
        if req_file.exists():
            print(f"Installing dependencies from {req_file}...")
            get_ipython().system(f"pip install -q -r {req_file}")
        break

# In remote Kaggle environment, if server/voices packages are not present locally:
if not any((Path(p) / "server").is_dir() and (Path(p) / "voices").is_dir() for p in sys.path):
    print("Remote Kaggle environment detected: cloning audiogen repository...")
    os.environ["GIT_TERMINAL_PROMPT"] = "0"
    get_ipython().system(
        "git clone https://github.com/lovishgoyal145/audiogen.git /kaggle/working/audiogen"
    )
    remote_repo = Path("/kaggle/working/audiogen")
    if remote_repo.is_dir():
        if str(remote_repo) not in sys.path:
            sys.path.insert(0, str(remote_repo))
        if (remote_repo / "src").is_dir() and str(remote_repo / "src") not in sys.path:
            sys.path.insert(0, str(remote_repo / "src"))
        req_file = remote_repo / "requirements.txt"
        if req_file.exists():
            print(f"Installing dependencies from {req_file}...")
            get_ipython().system(f"pip install -q -r {req_file}")

# Ensure cloudflared binary is available
if not shutil.which("cloudflared"):
    print("Installing cloudflared binary...")
    get_ipython().system("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    get_ipython().system("chmod +x /tmp/cloudflared")
    get_ipython().system("mv /tmp/cloudflared /usr/local/bin/cloudflared 2>/dev/null || cp /tmp/cloudflared /usr/bin/cloudflared 2>/dev/null || true")



In [ ]:
# Cell 2: Start FastAPI Speech Synthesis Server
import threading
import time
import uvicorn
from server.app import app

# Configure uvicorn for non-blocking execution on port 17000
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=17000,
    log_level="info",
)
server = uvicorn.Server(config)
server_thread = threading.Thread(
    target=server.run,
    daemon=True,
    name="UvicornServerThread",
)
server_thread.start()

# Wait briefly for socket binding
time.sleep(1.5)
print(f"FastAPI server running in background thread on port 17000.")


In [ ]:
# Load Registry & Server Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["TUNNEL_REGISTRY_WEBHOOK_URL"] = secrets.get_secret("TUNNEL_REGISTRY_WEBHOOK_URL")
os.environ["SERVER_BEARER_TOKEN"] = secrets.get_secret("SERVER_BEARER_TOKEN")


In [ ]:
# Cell 3: Launch Cloudflare Tunnel & Publish URL
import os
from server.tunnel import start_tunnel
from server.registry import publish_tunnel_url

# Start non-blocking Cloudflare tunnel to local port 17000
tunnel, tunnel_url = start_tunnel(
    target_url="http://localhost:17000",
    startup_timeout_seconds=30.0,
)
print(f"Cloudflare tunnel online: {tunnel_url}")

# Publish tunnel URL and bearer token to registry webhook
token = (
    os.environ.get("SERVER_BEARER_TOKEN")
    or os.environ.get("SHARED_SECRET")
    or ""
)
webhook_url = os.environ.get("TUNNEL_REGISTRY_WEBHOOK_URL")

if webhook_url and token:
    try:
        result = publish_tunnel_url(tunnel_url=tunnel_url, secret=token)
        print(f"Successfully published tunnel URL to registry: {result}")
    except Exception as exc:
        print(f"Warning: Failed to publish tunnel URL to registry: {exc}")
else:
    print("Notice: TUNNEL_REGISTRY_WEBHOOK_URL or secret token not set; skipping webhook publication.")


In [ ]:
# Cell 4: Start Idle Watchdog Daemon
from server.watchdog import IdleWatchdog

# Initialize watchdog (default 600s timeout, touches on requests)
watchdog = IdleWatchdog(idle_timeout_seconds=600.0, check_interval_seconds=1.0)
app.state.watchdog = watchdog
watchdog.start()
print(f"Idle watchdog started (timeout: {watchdog.idle_timeout_seconds}s).")


In [ ]:
# Cell 5: Main Execution Loop
import time

print("AudioGen interactive GPU server is online and awaiting requests.")
print("Session will automatically shut down after 600s of inactivity.")

try:
    while True:
        time.sleep(1.0)
except KeyboardInterrupt:
    print("Server interrupted. Stopping...")
